# 16 - Fine hyperparameter grid + blend-weight sweep (LB-testable submissions)

Motivation: a competitor scored **0.85718**, ~+0.0026 above our SVM anchor (0.85456).
That gap is within the public-LB sampling noise we've already seen in our own runs
(0.85264 / 0.85392 / 0.85456), so it is most likely a luckier draw of a ~0.855 model -
but we have spare submissions, so we probe the two cheap, legitimate levers here.

### Part A - finer C/gamma grid (result: FLAT)
A finer search than the original tuning. Best point `C=10, gamma=0.012` = **0.8415** vs
current `C=12, gamma=0.012` = 0.8413 -> +0.0002, pure noise. The SVM was already optimal.
We still emit a submission at the grid-best so you can confirm it's noise-identical.

### Part B - blend-weight sweep (the LB probe)
100% SVM = 0.85456 on LB but 50/50 SVM+CatBoost = 0.84736, so the optimum (if any) is a
*light* CatBoost dose. We emit submissions at SVM weight 0.90 / 0.85 / 0.80 for you to test.
Expected private gain ~0, but the public LB is the only judge of blend weight, and you can
afford the probes. Keep `svm_iterimpute` (0.85456) as the graded pick unless a probe beats
it by clearly more than ~0.002.

## Step 0 - Imports and data

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, numpy as np, pandas as pd
from sklearn.experimental import enable_iterative_imputer   # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score
from catboost import CatBoostClassifier

TRAIN_PATH, TEST_PATH, OUT_DIR = '../train.csv', '../test.csv', '../outputs'
train = pd.read_csv(TRAIN_PATH); test = pd.read_csv(TEST_PATH)
FEATURES = [c for c in train.columns if c not in ('id','sleep_stage')]
X, y, Xtest = train[FEATURES], train['sleep_stage'].to_numpy(), test[FEATURES]
def imputer(): return IterativeImputer(estimator=BayesianRidge(), max_iter=10, random_state=42)
print('train', train.shape, '| test', test.shape)

## Part A - finer C/gamma grid (screening on a single iterative imputation)
For speed we impute the full train once and scale once (screening only); the winner is
validated below with the real per-fold pipeline. Result is a flat plateau ~0.840-0.8415.

In [ ]:
Ximp = pd.DataFrame(imputer().fit_transform(X), columns=FEATURES)
Xs = StandardScaler().fit_transform(Ximp)
cv = StratifiedKFold(5, shuffle=True, random_state=42)
def sc(C, g):
    o = cross_val_predict(SVC(kernel='rbf', C=C, gamma=g, random_state=42), Xs, y, cv=cv, method='predict', n_jobs=5)
    return f1_score(y, o, average='macro')
Cs=[6,8,10,12,14,18,24]; Gs=[0.006,0.008,0.010,0.012,0.015,0.020,0.028]
best=(0,None,None)
print('C\\g   '+'  '.join('%.3f'%g for g in Gs))
for C in Cs:
    row=[sc(C,g) for g in Gs]
    for g,s in zip(Gs,row):
        if s>best[0]: best=(s,C,g)
    print('%4d  '%C+'  '.join('%.4f'%v for v in row), flush=True)
print('\nBEST CV=%.4f at C=%s gamma=%s  (current C=12,gamma=0.012 -> ~0.8413)'%best)
GRID_C, GRID_G = best[1], best[2]

## Part A submission - SVM at the grid-best C/gamma (real per-fold pipeline)
Writes `svm_gridbest_submission.csv`. Expect it to score ~0.855, noise-identical to the anchor.

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
svm_grid = make_pipeline(imputer(), StandardScaler(),
                         SVC(kernel='rbf', C=GRID_C, gamma=GRID_G, probability=True, random_state=42))
svm_grid.fit(X, y)
pred = svm_grid.predict(Xtest).astype(int)
sub = pd.DataFrame({'id': test['id'], 'sleep_stage': pred})
sub.to_csv(os.path.join(OUT_DIR,'svm_gridbest_submission.csv'), index=False)
print('wrote svm_gridbest_submission.csv  C=%s gamma=%s  dist=%s'
      % (GRID_C, GRID_G, dict(sub.sleep_stage.value_counts().sort_index())))

## Part B - blend-weight sweep submissions
Fit the anchor SVM (C=12, gamma=0.012) and CatBoost once, then emit one submission per
SVM weight. `predict_proba` is averaged: `w*SVM + (1-w)*CatBoost`.

In [ ]:
svm = make_pipeline(imputer(), StandardScaler(),
                    SVC(kernel='rbf', C=12, gamma=0.012, probability=True, random_state=42)).fit(X, y)
cat = make_pipeline(imputer(),
                    CatBoostClassifier(loss_function='MultiClass', iterations=600, depth=7,
                        learning_rate=0.04, l2_leaf_reg=3, random_state=42, verbose=0, thread_count=-1)).fit(X, y)
p_svm = svm.predict_proba(Xtest)
p_cat = np.asarray(cat.predict_proba(Xtest))
for w in [0.90, 0.85, 0.80]:
    pred = (w*p_svm + (1-w)*p_cat).argmax(1).astype(int)
    sub = pd.DataFrame({'id': test['id'], 'sleep_stage': pred})
    name = 'svm_cat_blend_w%02d_submission.csv' % int(w*100)
    sub.to_csv(os.path.join(OUT_DIR, name), index=False)
    print('wrote %-34s SVM weight %.2f  dist=%s' % (name, w, dict(sub.sleep_stage.value_counts().sort_index())))

## Takeaways
- **Part A (hyperparameters): flat.** The SVM was already at its optimum; the +0.0002
  'best' is noise. Rules out tuning as the source of a competitor's edge.
- **Part B (blend weight): the only LB-testable lever left.** Test w=0.90/0.85/0.80 on the
  public board. If one beats 0.85456 by clearly more than ~0.002, it may be a real (small)
  decorrelation gain; if they scatter around 0.855, it confirms the ceiling and the
  competitor's 0.85718 was a luckier public draw.
- **Graded pick stays `svm_iterimpute_submission.csv` (0.85456)** unless a probe clearly wins.